# Gibbs MCEM for the network Ising-logistic model

This notebook implements a practical Monte Carlo EM scheme for the model
$$
P_{\beta,\gamma}(\sigma\mid X,G)\propto
\exp\!\left(
\beta\sum_{(i,j)\in E}\sigma_i\sigma_j
+
2\sum_{i=1}^n \sigma_i X_i^\top\gamma
\right),
\qquad \sigma_i\in\{-1,1\},
$$
with partially observed labels.

The E-step uses Gibbs sampling on the unlabeled nodes, conditional on the observed labels.

The M-step maximizes a **lasso-penalized Monte Carlo logistic pseudolikelihood** using the Gibbs samples from the E-step.

The code is organized so that each major function sits in its own code cell.

Assumptions:
- labels are eventually converted to `{-1, +1}` for observed nodes and `0` for missing nodes;
- the graph is treated as **undirected** in the Ising term;
- if you want an intercept, add a column of ones to `X` before fitting.


In [ ]:
from pathlib import Path
import warnings

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.optimize import minimize
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message="IProgress not found.*")


In [ ]:
def set_random_seed(seed: int = 0):
    """Return a NumPy random generator with a fixed seed."""
    return np.random.default_rng(seed)

In [ ]:
def default_elliptic_root() -> Path:
    """Find the local EllipticBitcoinDataset directory from project or script cwd."""
    cwd = Path.cwd()
    candidates = [
        cwd / "data" / "EllipticBitcoinDataset",
        cwd.parent / "data" / "EllipticBitcoinDataset",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def load_elliptic_from_pyg(root=None):
    """Load the local PyG Elliptic Bitcoin dataset.

    PyG labels in this repository are encoded as:
    0 = licit, 1 = illicit, 2 = unknown.
    """
    from torch_geometric.datasets import EllipticBitcoinDataset

    root = default_elliptic_root() if root is None else Path(root)
    dataset = EllipticBitcoinDataset(root=str(root))
    data = dataset[0]

    X = data.x.cpu().numpy().astype(float)
    edge_index = data.edge_index.cpu().numpy().astype(int)
    y_raw = data.y.cpu().numpy()

    print(f"Loaded {root}")
    print(data)
    return X, edge_index, y_raw, data


In [ ]:
def normalize_labels(y_raw,
                     positive_values=(1,),
                     negative_values=(-1,),
                     unknown_values=(0, 2)):
    # \"\"\"Convert raw labels into {-1, +1, 0} where 0 means 'unobserved'.

    # Parameters
    # ----------
    # y_raw : array-like
    #     Raw label vector.
    # positive_values : tuple
    #     Values that should map to +1.
    # negative_values : tuple
    #     Values that should map to -1.
    # unknown_values : tuple
    #     Values that should map to 0.

    # Returns
    # -------
    # y_pm1 : ndarray, shape (n,)
    #     Label vector in {-1, 0, +1}.
    # observed_mask : ndarray[bool], shape (n,)
    #     True where the label is observed.
    # unlabeled_mask : ndarray[bool], shape (n,)
    #     True where the label is missing.
    # \"\"\"
    y = np.asarray(y_raw).copy()
    y_pm1 = np.zeros(len(y), dtype=np.int8)

    pos_set = set(positive_values)
    neg_set = set(negative_values)
    unk_set = set(v for v in unknown_values if not (isinstance(v, float) and np.isnan(v)))

    for i, val in enumerate(y):
        if isinstance(val, float) and np.isnan(val):
            y_pm1[i] = 0
        elif val in pos_set:
            y_pm1[i] = 1
        elif val in neg_set:
            y_pm1[i] = -1
        elif val in unk_set:
            y_pm1[i] = 0
        else:
            raise ValueError(
                f"Encountered label value {val!r} that is not covered by "
                "positive_values / negative_values / unknown_values."
            )

    observed_mask = y_pm1 != 0
    unlabeled_mask = ~observed_mask
    return y_pm1, observed_mask, unlabeled_mask

In [ ]:
def standardize_features(X):
    # \"\"\"Standardize features columnwise.

    # Returns
    # -------
    # X_std : ndarray
    #     Standardized feature matrix.
    # scaler : StandardScaler
    #     Fitted scaler object.
    # \"\"\"
    X = np.asarray(X, dtype=float)
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    return X_std, scaler

In [ ]:
def build_symmetric_adjacency(num_nodes: int,
                              edge_index,
                              remove_self_loops: bool = True):
    # \"\"\"Build a symmetric CSR adjacency matrix from an edge list.

    # Parameters
    # ----------
    # num_nodes : int
    #     Number of nodes.
    # edge_index : array-like, shape (2, m) or (m, 2)
    #     Edge list.
    # remove_self_loops : bool
    #     If True, drop diagonal edges.

    # Returns
    # -------
    # A : scipy.sparse.csr_matrix, shape (n, n)
    #     Symmetric 0/1 adjacency matrix.
    # \"\"\"
    edge_index = np.asarray(edge_index, dtype=int)
    if edge_index.ndim != 2:
        raise ValueError("edge_index must be a 2D array.")

    if edge_index.shape[0] == 2:
        src, dst = edge_index[0], edge_index[1]
    elif edge_index.shape[1] == 2:
        src, dst = edge_index[:, 0], edge_index[:, 1]
    else:
        raise ValueError("edge_index must have shape (2, m) or (m, 2).")

    data = np.ones(len(src), dtype=np.float64)
    A = sp.coo_matrix((data, (src, dst)), shape=(num_nodes, num_nodes))
    A = A + A.T
    A = A.tocsr()

    if remove_self_loops:
        A.setdiag(0)
        A.eliminate_zeros()

    A.data[:] = 1.0
    return A


In [ ]:
def initialize_parameters_via_logistic(X,
                                       y_pm1,
                                       l2_C: float = 1.0,
                                       max_iter: int = 1000,
                                       random_state: int = 0):
    # \"\"\"Initialize beta, gamma, and the latent labels using logistic regression.

    # Parameters
    # ----------
    # X : ndarray, shape (n, p)
    # y_pm1 : ndarray, shape (n,)
    #     Labels in {-1, 0, +1}; 0 means missing.
    # l2_C : float
    #     Inverse L2 regularization strength for sklearn logistic regression.
    # max_iter : int
    #     Max iterations for sklearn logistic regression.
    # random_state : int
    #     Random seed for sklearn.

    # Returns
    # -------
    # beta0 : float
    #     Initialized as 0.
    # gamma0 : ndarray, shape (p,)
    #     Since our model with beta = 0 has
    #         P(sigma_i = 1 | X_i) = sigmoid(4 X_i^T gamma),
    #     we divide sklearn's coefficient vector by 4.
    # sigma0 : ndarray, shape (n,)
    #     Initial complete label vector in {-1, +1}, where unlabeled entries are
    #     filled using the logistic fit.
    # init_proba : ndarray, shape (n,)
    #     Initial probabilities P(sigma_i = 1 | X_i) from the logistic fit.
    # init_model : fitted sklearn LogisticRegression
    #     The underlying logistic model.
    # \"\"\"
    X = np.asarray(X, dtype=float)
    y_pm1 = np.asarray(y_pm1, dtype=np.int8)

    observed_mask = y_pm1 != 0
    if observed_mask.sum() == 0:
        raise ValueError("No observed labels found.")

    y_binary = (y_pm1[observed_mask] == 1).astype(int)
    if np.unique(y_binary).size < 2:
        raise ValueError("Observed labels contain only one class; logistic initialization is not possible.")

    model = LogisticRegression(
        C=l2_C,
        fit_intercept=False,
        solver="lbfgs",
        max_iter=max_iter,
        random_state=random_state,
    )
    model.fit(X[observed_mask], y_binary)

    logits = model.decision_function(X)
    init_proba = 1.0 / (1.0 + np.exp(-logits))
    gamma0 = model.coef_.reshape(-1) / 4.0
    beta0 = 0.0

    sigma0 = np.where(init_proba >= 0.5, 1, -1).astype(np.int8)
    sigma0[observed_mask] = y_pm1[observed_mask]

    return beta0, gamma0, sigma0, init_proba, model


In [ ]:
def compute_local_field(node_index: int, sigma, A, X, beta: float, gamma):
    # \"\"\"Compute the local field
    #     h_i = beta * sum_{j ~ i} sigma_j + 2 X_i^T gamma
    # for a single node.
    # \"\"\"
    row_start = A.indptr[node_index]
    row_end = A.indptr[node_index + 1]
    neighbors = A.indices[row_start:row_end]
    neighbor_sum = sigma[neighbors].sum()
    return beta * neighbor_sum + 2.0 * float(X[node_index] @ gamma)

In [ ]:
def gibbs_sweep_inplace(sigma,
                       neighbor_sum,
                       unlabeled_idx,
                       A,
                       X,
                       beta: float,
                       gamma,
                       rng):
    # \"\"\"Perform one in-place Gibbs sweep over the unlabeled nodes.

    # Parameters
    # ----------
    # sigma : ndarray, shape (n,)
    #     Current complete label vector in {-1, +1}. Modified in place.
    # neighbor_sum : ndarray, shape (n,)
    #     Current vector A @ sigma. Modified in place.
    # unlabeled_idx : ndarray, shape (n_unlabeled,)
    #     Indices of unlabeled nodes.
    # A : csr_matrix
    #     Symmetric adjacency.
    # X : ndarray, shape (n, p)
    #     Feature matrix.
    # beta : float
    # gamma : ndarray, shape (p,)
    # rng : np.random.Generator

    # Returns
    # -------
    # sigma : ndarray
    #     Updated complete labels.
    # neighbor_sum : ndarray
    #     Updated neighbor sums.
    # \"\"\"
    order = rng.permutation(unlabeled_idx)

    linear_term = 2.0 * (X @ gamma)

    for i in order:
        h_i = beta * neighbor_sum[i] + linear_term[i]
        p_plus = 1.0 / (1.0 + np.exp(-2.0 * h_i))
        new_val = 1 if rng.random() < p_plus else -1
        old_val = sigma[i]

        if new_val != old_val:
            delta = new_val - old_val
            sigma[i] = new_val

            row_start = A.indptr[i]
            row_end = A.indptr[i + 1]
            neighbors = A.indices[row_start:row_end]
            neighbor_sum[neighbors] += delta

    return sigma, neighbor_sum

In [ ]:
def draw_posterior_samples(sigma_start,
                           observed_mask,
                           y_pm1,
                           A,
                           X,
                           beta: float,
                           gamma,
                           num_burnin: int = 50,
                           num_samples: int = 20,
                           thinning: int = 5,
                           seed: int = 0):
    # \"\"\"Draw posterior samples of the unlabeled spins by Gibbs sampling.

    # Observed spins are held fixed throughout.

    # Parameters
    # ----------
    # sigma_start : ndarray, shape (n,)
    #     Initial complete label vector in {-1, +1}.
    # observed_mask : ndarray[bool], shape (n,)
    #     True where labels are observed.
    # y_pm1 : ndarray, shape (n,)
    #     Observed labels in {-1, 0, +1}.
    # A : csr_matrix
    # X : ndarray, shape (n, p)
    # beta : float
    # gamma : ndarray, shape (p,)
    # num_burnin : int
    # num_samples : int
    # thinning : int
    # seed : int

    # Returns
    # -------
    # samples : ndarray, shape (num_samples, n)
    #     Posterior samples of the complete spin vector.
    # sigma_last : ndarray, shape (n,)
    #     Last state of the chain, useful as a warm start for the next EM step.
    # \"\"\"
    rng = np.random.default_rng(seed)

    sigma = np.asarray(sigma_start, dtype=np.int8).copy()
    sigma[observed_mask] = y_pm1[observed_mask]

    unlabeled_idx = np.where(~observed_mask)[0]
    neighbor_sum = np.asarray(A @ sigma, dtype=np.float64).reshape(-1)

    for _ in range(num_burnin):
        sigma, neighbor_sum = gibbs_sweep_inplace(
            sigma=sigma,
            neighbor_sum=neighbor_sum,
            unlabeled_idx=unlabeled_idx,
            A=A,
            X=X,
            beta=beta,
            gamma=gamma,
            rng=rng,
        )
        sigma[observed_mask] = y_pm1[observed_mask]

    samples = []
    for _ in range(num_samples):
        for _ in range(thinning):
            sigma, neighbor_sum = gibbs_sweep_inplace(
                sigma=sigma,
                neighbor_sum=neighbor_sum,
                unlabeled_idx=unlabeled_idx,
                A=A,
                X=X,
                beta=beta,
                gamma=gamma,
                rng=rng,
            )
            sigma[observed_mask] = y_pm1[observed_mask]

        samples.append(sigma.copy())

    samples = np.asarray(samples, dtype=np.int8)
    return samples, sigma.copy()

In [ ]:
def estimate_posterior_means(samples, observed_mask, y_pm1):
    # \"\"\"Estimate posterior means E[sigma_i | observed data] from Gibbs samples.

    # Parameters
    # ----------
    # samples : ndarray, shape (M, n)
    # observed_mask : ndarray[bool], shape (n,)
    # y_pm1 : ndarray, shape (n,)

    # Returns
    # -------
    # posterior_mean : ndarray, shape (n,)
    #     Monte Carlo estimate of E[sigma_i | observed data].
    # posterior_prob_plus : ndarray, shape (n,)
    #     Monte Carlo estimate of P(sigma_i = +1 | observed data).
    # \"\"\"
    posterior_mean = samples.mean(axis=0)
    posterior_mean = posterior_mean.astype(float)
    posterior_mean[observed_mask] = y_pm1[observed_mask]
    posterior_prob_plus = 0.5 * (posterior_mean + 1.0)
    return posterior_mean, posterior_prob_plus

In [ ]:
def build_mc_logistic_pseudolikelihood_design(samples, A, X):
    """Build the conditional-logistic design matrix for the MC pseudolikelihood.

    For spins sigma_i in {-1, +1},
        P(sigma_i = +1 | rest) = sigmoid(2 beta * neighbor_sum_i + 4 X_i^T gamma).
    Hence each sampled node contributes one logistic-regression row with features
        [2 * neighbor_sum_i, 4 * X_i]
    and binary target 1{sigma_i = +1}.
    """
    samples = np.asarray(samples, dtype=np.int8)
    X = np.asarray(X, dtype=float)
    num_samples, num_nodes = samples.shape
    num_features = X.shape[1]

    design = np.empty((num_samples * num_nodes, num_features + 1), dtype=float)
    target = np.empty(num_samples * num_nodes, dtype=int)

    feature_block = 4.0 * X
    for sample_id, sigma in enumerate(samples):
        start = sample_id * num_nodes
        end = start + num_nodes
        neighbor_sum = np.asarray(A @ sigma.astype(float), dtype=float).reshape(-1)
        design[start:end, 0] = 2.0 * neighbor_sum
        design[start:end, 1:] = feature_block
        target[start:end] = (sigma == 1).astype(int)

    return design, target


In [ ]:
def m_step_optimize(samples,
                    A,
                    X,
                    beta_init: float,
                    gamma_init,
                    lasso_C: float = 1.0,
                    beta_nonnegative: bool = True,
                    maxiter: int = 500,
                    seed: int = 0):
    """Run the M-step with lasso-logistic regression on MC pseudolikelihood rows.

    lasso_C is sklearn's inverse regularization strength: smaller values mean
    a stronger L1/lasso penalty.
    """
    design, target = build_mc_logistic_pseudolikelihood_design(samples=samples, A=A, X=X)

    model = LogisticRegression(
        l1_ratio=1.0,
        C=lasso_C,
        fit_intercept=False,
        solver="saga",
        max_iter=maxiter,
        tol=1e-3,
        random_state=seed,
    )
    model.fit(design, target)

    coef = model.coef_.reshape(-1)
    beta_hat = float(coef[0])
    gamma_hat = np.asarray(coef[1:], dtype=float)

    # Keep the homophily parameter nonnegative when requested. If the fitted
    # conditional logistic model wants a negative beta, project it to zero.
    if beta_nonnegative and beta_hat < 0.0:
        beta_hat = 0.0

    return beta_hat, gamma_hat, model


In [ ]:
def predict_labels_from_posterior_mean(posterior_mean):
    # \"\"\"Convert posterior means into hard labels and probabilities.

    # Returns
    # -------
    # hard_labels : ndarray, shape (n,)
    #     Values in {-1, +1}.
    # prob_plus : ndarray, shape (n,)
    #     Values in [0, 1].
    # \"\"\"
    posterior_mean = np.asarray(posterior_mean, dtype=float)
    hard_labels = np.where(posterior_mean >= 0.0, 1, -1).astype(np.int8)
    prob_plus = 0.5 * (posterior_mean + 1.0)
    return hard_labels, prob_plus

In [ ]:
def run_gibbs_mcem(y_pm1,
                   X,
                   A,
                   num_em_steps: int = 10,
                   num_burnin: int = 50,
                   num_samples: int = 20,
                   thinning: int = 5,
                   lasso_C: float = 1.0,
                   init_logistic_C: float = 1.0,
                   beta_nonnegative: bool = True,
                   optimizer_maxiter: int = 200,
                   seed: int = 0):
    # \"\"\"Run Gibbs-MCEM with a pseudolikelihood M-step.

    # Parameters
    # ----------
    # y_pm1 : ndarray, shape (n,)
    #     Labels in {-1, 0, +1}; 0 means missing.
    # X : ndarray, shape (n, p)
    # A : csr_matrix, shape (n, n)
    # num_em_steps : int
    # num_burnin : int
    # num_samples : int
    # thinning : int
    # lasso_C : float
    #     Inverse L1 regularization strength for the lasso-logistic pseudolikelihood M-step.
    # init_logistic_C : float
    # beta_nonnegative : bool
    # optimizer_maxiter : int
    # seed : int

    # Returns
    # -------
    # results : dict
    #     Dictionary containing fitted parameters, posterior summaries, and
    #     iteration history.
    # \"\"\"
    y_pm1 = np.asarray(y_pm1, dtype=np.int8)
    X = np.asarray(X, dtype=float)

    observed_mask = y_pm1 != 0
    unlabeled_mask = ~observed_mask

    beta, gamma, sigma_current, init_proba, init_model = initialize_parameters_via_logistic(
        X=X,
        y_pm1=y_pm1,
        l2_C=init_logistic_C,
        random_state=seed,
    )

    history = []
    posterior_mean = sigma_current.astype(float)
    posterior_prob_plus = 0.5 * (posterior_mean + 1.0)

    for t in range(num_em_steps):
        samples, sigma_last = draw_posterior_samples(
            sigma_start=sigma_current,
            observed_mask=observed_mask,
            y_pm1=y_pm1,
            A=A,
            X=X,
            beta=beta,
            gamma=gamma,
            num_burnin=num_burnin,
            num_samples=num_samples,
            thinning=thinning,
            seed=seed + 1000 * (t + 1),
        )

        posterior_mean, posterior_prob_plus = estimate_posterior_means(
            samples=samples,
            observed_mask=observed_mask,
            y_pm1=y_pm1,
        )

        beta_new, gamma_new, m_step_model = m_step_optimize(
            samples=samples,
            A=A,
            X=X,
            beta_init=beta,
            gamma_init=gamma,
            lasso_C=lasso_C,
            beta_nonnegative=beta_nonnegative,
            maxiter=optimizer_maxiter,
            seed=seed + 2000 * (t + 1),
        )

        sigma_current = np.where(posterior_mean >= 0.0, 1, -1).astype(np.int8)
        sigma_current[observed_mask] = y_pm1[observed_mask]

        m_step_iterations = int(np.max(m_step_model.n_iter_))
        m_step_converged = m_step_iterations < optimizer_maxiter

        history.append({
            "iteration": t + 1,
            "beta": beta_new,
            "gamma_l2_norm": float(np.linalg.norm(gamma_new)),
            "m_step_converged": bool(m_step_converged),
            "m_step_iterations": m_step_iterations,
            "m_step_message": "sklearn lasso LogisticRegression fit",
            "mean_posterior_prob_unlabeled": (
                float(posterior_prob_plus[unlabeled_mask].mean())
                if unlabeled_mask.any() else np.nan
            ),
        })

        beta, gamma = beta_new, gamma_new

    hard_labels, hard_prob_plus = predict_labels_from_posterior_mean(posterior_mean)

    results = {
        "beta": beta,
        "gamma": gamma,
        "posterior_mean": posterior_mean,
        "posterior_prob_plus": posterior_prob_plus,
        "hard_labels": hard_labels,
        "observed_mask": observed_mask,
        "unlabeled_mask": unlabeled_mask,
        "history": history,
        "initial_logistic_prob_plus": init_proba,
        "initial_logistic_model": init_model,
        "last_complete_state": sigma_last,
    }
    return results


## Elliptic pilot run

The full graph has 203,769 nodes, so start with a connected pilot subgraph before scaling up.  The Ising interaction below treats the transaction graph as undirected by symmetrizing `edge_index`, while the raw Elliptic edges remain directed Bitcoin flows.


In [ ]:
# Elliptic connected-component usage
# ----------------------------------
# MAX_NODES = None uses the entire selected connected component.
# COMPONENT_RANK = 1 is the largest connected component.

MAX_NODES = None
NUM_FEATURES = 94
COMPONENT_RANK = 1


def select_connected_component_nodes(data, max_nodes=MAX_NODES, component_rank=COMPONENT_RANK):
    """Select all nodes from one weakly connected component, or a BFS sample if capped."""
    graph = nx.Graph()
    graph.add_nodes_from(range(data.num_nodes))
    graph.add_edges_from(zip(data.edge_index[0].tolist(), data.edge_index[1].tolist()))

    components = sorted(nx.connected_components(graph), key=len, reverse=True)
    component = set(components[component_rank - 1])
    labels = data.y.cpu().numpy()

    if max_nodes is None or max_nodes >= len(component):
        return np.array(sorted(component), dtype=int), len(component)

    # Start from an illicit node when possible so small capped runs contain both classes.
    illicit_nodes = [node for node in component if labels[node] == 1]
    seed = min(illicit_nodes) if illicit_nodes else min(component)

    sampled = []
    seen = {seed}
    queue = [seed]
    while queue and len(sampled) < max_nodes:
        current = queue.pop(0)
        sampled.append(current)
        for neighbor in sorted(graph.neighbors(current)):
            if neighbor in component and neighbor not in seen:
                seen.add(neighbor)
                queue.append(neighbor)
            if len(sampled) + len(queue) >= max_nodes:
                break

    return np.array(sampled, dtype=int), len(component)


def remap_edge_index_to_nodes(edge_index, nodes):
    """Keep induced-subgraph edges and remap original node ids to 0..n_sub-1."""
    nodes = np.asarray(nodes, dtype=int)
    max_index = int(max(edge_index.max(), nodes.max()))
    old_to_new = np.full(max_index + 1, -1, dtype=int)
    old_to_new[nodes] = np.arange(len(nodes))

    src = np.asarray(edge_index[0], dtype=int)
    dst = np.asarray(edge_index[1], dtype=int)
    keep = (old_to_new[src] >= 0) & (old_to_new[dst] >= 0)
    return np.vstack((old_to_new[src[keep]], old_to_new[dst[keep]]))


X, edge_index, y_raw, data_obj = load_elliptic_from_pyg()

nodes, full_component_size = select_connected_component_nodes(
    data_obj,
    max_nodes=MAX_NODES,
    component_rank=COMPONENT_RANK,
)
X_component = X[nodes, :NUM_FEATURES]
y_raw_component = y_raw[nodes]
edge_index_component = remap_edge_index_to_nodes(edge_index, nodes)

# Correct PyG Elliptic label mapping for this repo:
# illicit -> +1, licit -> -1, unknown -> 0/missing.
y_pm1, observed_mask, unlabeled_mask = normalize_labels(
    y_raw_component,
    positive_values=(1,),
    negative_values=(0,),
    unknown_values=(2,),
)

if len(np.unique(y_pm1[observed_mask])) < 2:
    raise ValueError("Selected component has only one observed class. Choose a larger/different component.")

X_std, scaler = standardize_features(X_component)
A = build_symmetric_adjacency(num_nodes=X_std.shape[0], edge_index=edge_index_component)

print(f"Component rank: {COMPONENT_RANK}")
print(f"Full component size: {full_component_size:,}")
print(f"Nodes used: {X_std.shape[0]:,}")
print(f"Directed edges used: {edge_index_component.shape[1]:,}")
print(f"Undirected Ising edges: {A.nnz // 2:,}")
print(
    "Labels:",
    f"licit={(y_pm1 == -1).sum():,}",
    f"illicit={(y_pm1 == 1).sum():,}",
    f"unknown={(y_pm1 == 0).sum():,}",
)

results = run_gibbs_mcem(
    y_pm1=y_pm1,
    X=X_std,
    A=A,
    num_em_steps=2,
    num_burnin=5,
    num_samples=4,
    thinning=2,
    lasso_C=1.0,
    init_logistic_C=1.0,
    beta_nonnegative=True,
    optimizer_maxiter=500,
    seed=305,
)

print("beta_hat =", results["beta"])
print("first 10 posterior illicit probabilities =", results["posterior_prob_plus"][:10])
print("history =", results["history"])


## Original vs predicted network

This uses the same selected connected component and layout for both panels. Gray nodes on the left are originally unknown labels; the right panel shows the MCEM posterior hard prediction for every node.


In [ ]:
def plot_original_vs_predicted_network(edge_index_component, y_raw_component, results, output_path=None):
    graph = nx.Graph()
    graph.add_nodes_from(range(len(y_raw_component)))
    graph.add_edges_from(zip(edge_index_component[0].tolist(), edge_index_component[1].tolist()))

    pos = nx.spring_layout(graph, seed=305, k=0.06, iterations=60)

    original_colors = {
        0: "#2a9d8f",  # licit
        1: "#e63946",  # illicit
        2: "#8d99ae",  # unknown
    }
    predicted_colors = {
        -1: "#2a9d8f",  # predicted licit
        1: "#e63946",   # predicted illicit
    }

    hard_labels = results["hard_labels"]
    original_node_colors = [original_colors[int(label)] for label in y_raw_component]
    predicted_node_colors = [predicted_colors[int(label)] for label in hard_labels]

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
    panels = [
        (axes[0], original_node_colors, "Original labels", "gray = unknown"),
        (axes[1], predicted_node_colors, "Predicted labels", "posterior hard labels"),
    ]

    for ax, colors, title, subtitle in panels:
        nx.draw_networkx_edges(graph, pos, ax=ax, alpha=0.08, width=0.35, edge_color="#4a4e69")
        nx.draw_networkx_nodes(
            graph,
            pos,
            ax=ax,
            node_color=colors,
            node_size=max(3, min(12, 18000 / graph.number_of_nodes())),
            linewidths=0,
        )
        ax.set_title(title + "\n" + subtitle, fontsize=13)
        ax.axis("off")

    legend_handles = [
        plt.Line2D([0], [0], marker="o", color="w", label="licit", markerfacecolor="#2a9d8f", markersize=8),
        plt.Line2D([0], [0], marker="o", color="w", label="illicit", markerfacecolor="#e63946", markersize=8),
        plt.Line2D([0], [0], marker="o", color="w", label="unknown original", markerfacecolor="#8d99ae", markersize=8),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=False)

    if output_path is not None:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=220, bbox_inches="tight")
        print(f"Saved figure to {output_path}")

    return fig, axes


fig, axes = plot_original_vs_predicted_network(
    edge_index_component=edge_index_component,
    y_raw_component=y_raw_component,
    results=results,
    output_path=default_elliptic_root().parents[1] / "figures" / "ising_original_vs_predicted_component.png",
)
plt.show()


## Practical remarks

1. This is a **Monte Carlo EM with a lasso-logistic pseudolikelihood M-step**, not exact full-likelihood EM.

2. On a graph as large as Elliptic, start with small values such as:
   - `num_em_steps = 3 to 5`
   - `num_burnin = 10 to 30`
   - `num_samples = 5 to 15`
   - `thinning = 2 to 5`

3. If you want an intercept, append a column of ones to `X` before standardization, or standardize first and then append the ones column.

4. Since the Ising interaction is meant to be symmetric, directed transaction edges are symmetrized in `build_symmetric_adjacency`.

5. The Gibbs sweep uses an in-place update of the vector `A @ sigma`, so each sweep is roughly linear in the number of edges.
